# Exploratory Data Analysis: Used Car Price Prediction

This notebook explores the real-world **CarDekho Indian Used Car Dataset** (`dataset/used_cars.csv`), inspects data quality, analyzes selling price distributions, studies depreciation factors, and summarizes findings for regression modeling.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_raw_dataset, inspect_dataset
from src.preprocessing import prepare_cleaned_dataframe

sns.set_theme(style='whitegrid')
df_raw = load_raw_dataset(PROJECT_ROOT / 'dataset' / 'used_cars.csv')
summary = inspect_dataset(df_raw)
print(f"Rows: {summary['rows']}, Columns: {summary['columns']}, Duplicates: {summary['duplicate_rows']}")
df_raw.head()

## 1. Data Types, Missing Values, and Price Summary

In [ ]:
display(df_raw.info())
print('\nMissing Values per Column:')
print(df_raw.isnull().sum())
print('\nSelling Price Summary (INR):')
print(df_raw['selling_price'].describe().round(2))

## 2. Data Cleaning & Feature Engineering
We remove exact duplicate listings, filter extreme odometer anomalies (> 500,000 km), extract numeric values from `mileage`, `engine`, and `max_power`, extract `brand` and `car_model`, and compute `vehicle_age` relative to 2020.

In [ ]:
df_clean, clean_report = prepare_cleaned_dataframe(df_raw)
print('Cleaning Report:', clean_report)
df_clean[['brand', 'car_model', 'vehicle_age', 'km_driven', 'fuel', 'transmission', 'mileage', 'engine', 'max_power', 'selling_price']].head()

## 3. Selling Price Distribution & Correlation Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(df_clean['selling_price'] / 1e5, bins=40, kde=True, ax=axes[0], color='#1D6FA5')
axes[0].set_title('Selling Price Distribution (INR Lakhs)')
axes[0].set_xlabel('Selling Price (Lakhs)')

corr_cols = ['selling_price', 'vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats']
sns.heatmap(df_clean[corr_cols].corr(), annot=True, fmt='.2f', cmap='Blues', ax=axes[1])
axes[1].set_title('Numerical Feature Correlations')
plt.tight_layout()
plt.show()

## 4. Depreciation & Categorical Comparisons

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
df_clean.groupby('vehicle_age')['selling_price'].mean().loc[:18].div(1e5).plot(marker='o', ax=axes[0], color='#0F2942')
axes[0].set_title('Mean Price vs Vehicle Age')
axes[0].set_ylabel('Price (INR Lakhs)')

df_clean.groupby('fuel')['selling_price'].mean().sort_values(ascending=False).div(1e5).plot(kind='bar', ax=axes[1], color='#1D6FA5')
axes[1].set_title('Mean Price by Fuel Type')

df_clean.groupby('transmission')['selling_price'].mean().sort_values(ascending=False).div(1e5).plot(kind='bar', ax=axes[2], color='#2E9CCA')
axes[2].set_title('Mean Price by Transmission')
plt.tight_layout()
plt.show()

## Key Takeaways
- **Right-skewed target**: Most listings are mass-market vehicles under ₹10 Lakhs, while luxury brands create a long upper tail up to ₹1 Crore.
- **Strongest numerical predictors**: `max_power` and `engine` correlate positively with price, whereas `vehicle_age` and `km_driven` show clear depreciation relationships.
- **Transmission & Fuel**: Automatic and Diesel vehicles exhibit higher mean resale prices than Manual and Petrol/CNG/LPG vehicles in this dataset.